# 01 · Tokenization Baselines

BPE entropy analysis; demonstrates why fixed-vocabulary tokenization fails as a concept boundary mechanism.

**Papers:** Vaswani 2017 ([1706.03762](https://arxiv.org/abs/1706.03762)), Mielke 2021 ([2112.10508](https://arxiv.org/abs/2112.10508))

In [ ]:
# pip install transformers matplotlib
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
print("Tokenizer loaded.")

## 1. Token length distributions by domain

In [ ]:
samples = {
    "casual":         "Hey how are you doing today? The weather is really nice.",
    "technical":      "The gradient of cross-entropy loss with respect to logits equals softmax(z) - y.",
    "code":           "def forward(self, x):
    return self.norm(self.attn(x) + x)",
    "named_entities": "Holger Schwenk and Paul-Ambroise Duquenne published LCM in December 2024.",
}
for name, text in samples.items():
    ids  = tokenizer.encode(text)
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"[{name}] {len(ids):3d} tokens")
    print("  " + " | ".join(toks[:18]))

## 2. Per-token entropy profile

High-entropy positions are natural concept boundaries.

In [ ]:
import torch
from transformers import GPT2LMHeadModel
model_lm = GPT2LMHeadModel.from_pretrained("gpt2").eval()
text = "The dynamic segmentation mechanism identifies semantic boundaries by measuring local dissimilarity."
ids_t = torch.tensor([tokenizer.encode(text)])
with torch.no_grad():
    logits = model_lm(ids_t).logits[0]
    probs  = torch.softmax(logits, dim=-1)
    H      = -(probs * torch.log(probs + 1e-9)).sum(-1).numpy()
tokens = tokenizer.convert_ids_to_tokens(ids_t[0].tolist())
colors = ["#C00000" if h > H.mean()+H.std() else "#2E75B6" if h > H.mean() else "#AAAAAA" for h in H]
fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(len(H)), H, color=colors)
ax.axhline(H.mean(), color="black", linestyle="--", lw=1, label="mean")
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Entropy (nats)")
ax.set_title("Per-token entropy — red = high-information transitions (ideal concept boundaries)")
ax.legend(); plt.tight_layout(); plt.savefig("../data/samples/entropy_profile.png", dpi=150); plt.show()
high = [tokens[i] for i in range(len(tokens)) if H[i] > H.mean() + H.std()]
print(f"High-entropy tokens: {high}")

## 3. BPE failure modes

In [ ]:
for term in ["Schwenk", "arXiv:2412.08821", "muP", "SONAR", "DLCM"]:
    toks = tokenizer.convert_ids_to_tokens(tokenizer.encode(term))
    print(f"  {term:25s} -> {toks}")

In [ ]:
sentences = {
    "English": "The cat sat on the mat.",
    "French":  "Le chat etait assis sur le tapis.",
    "German":  "Die Katze sass auf der Matte.",
    "Arabic":  "alqit jalasa ealaa alhasira.",
}
for lang, s in sentences.items():
    n = len(tokenizer.encode(s))
    print(f"  {lang:10s}: {n:3d} tokens | {s}")
print("
Same concept, different token cost. Concept-level modeling collapses this.")

## Summary

| | BPE Token | Semantic Concept |
|---|---|---|
| Boundary criterion | Corpus frequency | Semantic transition in latent space |
| Granularity | Fixed | Adaptive to information density |
| Language-agnostic | No | Yes (DLCM latent space) |
| Compute uniform | Yes (wasteful) | No (more compute at boundaries) |

**Takeaway:** BPE tokens are compression artifacts. DLCM boundary scoring (Eq. 6) places splits where the latent representation changes most.